In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob

import itertools
from pybedtools import BedTool

from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

from scipy.stats import ttest_ind

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2','IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
indir = f'{projdir}/pmds/dnmtools/rna_cluster_donor'
pmds = pd.read_csv(f'{indir}/merged/start_merged.filt_mcg.sorted.bed.gz', sep='\t', header=None)
pmds.index = [f'{x}_{y}' for x,y in zip(pmds[0], pmds[1])]
pmds.shape

In [ ]:
pmds.head()

In [ ]:
clusts

In [ ]:
indir = f'{projdir}/bed/pmds/rna_cluster_donor'

def load_allc(s):
    allc = pd.read_csv(f'{indir}/{s}.start_pmds_CGN-Both.full.bed.gz', sep='\t', header=None)
    return allc

with ProcessPoolExecutor() as executor:
    pmds_results = list(tqdm(executor.map(load_allc, clusts), total=len(clusts)))

In [ ]:
merged_mc = np.vstack([(x[3] / x[4]).values for x in pmds_results])
pmds_mc = pd.DataFrame(merged_mc, columns=pmds.index, index=clusts).T
pmds_mc.shape

In [ ]:
pmds_mc.head()

In [ ]:
pmds_mc.max()

In [ ]:
### ok nice
s = 'Start_C39'
pmds_mc[s].max()

In [ ]:
indir = f'{projdir}/pmds/dnmtools/rna_cluster_donor'
comp = pd.read_csv(f'{indir}/merged/start_merged.filt_mcg.comp.sorted.bed.gz', sep='\t', header=None)
comp.index = [f'{x}_{y}' for x,y in zip(comp[0], comp[1])]
comp.shape

In [ ]:
comp.head()

In [ ]:
indir = f'{projdir}/bed/pmds/rna_cluster_donor'

def load_allc(s):
    allc = pd.read_csv(f'{indir}/{s}.start_pmds_comp_CGN-Both.full.bed.gz', sep='\t', header=None)
    return allc

with ProcessPoolExecutor() as executor:
    comp_results = list(tqdm(executor.map(load_allc, clusts), total=len(clusts)))

In [ ]:
merged_mc = np.vstack([(x[3] / x[4]).values for x in comp_results])
comp_mc = pd.DataFrame(merged_mc, columns=comp.index, index=clusts).T
### need to figure out where there is nan?
comp_mc.dropna(inplace=True)
comp_mc.shape

In [ ]:
comp_mc.head()

### plot mcg

In [ ]:
pmds_plot = pmds_mc.melt(value_vars=pmds_mc.columns)
pmds_plot['cluster'] = [x.split('_')[0] for x in pmds_plot['variable']]
pmds_plot['donor'] = [x.split('_')[1] for x in pmds_plot['variable']]
pmds_plot.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(6, 5))

x_lab = 'cluster'
y_lab = 'value'
hue = 'donor'

ax = sns.boxplot(pmds_plot, x=x_lab, y=y_lab, showfliers=False,
                 hue=hue, order=tmp_order)
                 
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('', fontsize=12)
ax.set_title('', fontsize=20)
ax.grid(False)

ax.tick_params(axis='x', labelrotation=45)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
comp_plot = comp_mc.melt(value_vars=comp_mc.columns)
comp_plot['cluster'] = [x.split('_')[0] for x in comp_plot['variable']]
comp_plot['donor'] = [x.split('_')[1] for x in comp_plot['variable']]
comp_plot.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(6, 5))

x_lab = 'cluster'
y_lab = 'value'
hue = 'donor'

ax = sns.boxplot(comp_plot, x=x_lab, y=y_lab, showfliers=False,
                 hue=hue, order=tmp_order)
                 
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('', fontsize=12)
ax.set_title('', fontsize=20)
ax.grid(False)

ax.tick_params(axis='x', labelrotation=45)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
s = 'Start_C29'
ttest_ind(pmds_mc[s], comp_mc[s])

### I want to facet this

In [ ]:
pmds_plot['type'] = 'PMDs'
comp_plot['type'] = 'Non-PMDs'
merged = pd.concat([pmds_plot, comp_plot])
merged.shape

In [ ]:
merged

In [ ]:
fig, axes = plt.subplots(1, figsize=(6, 5))

x_lab = 'cluster'
y_lab = 'value'
hue = 'type'

ax = sns.boxplot(merged, x=x_lab, y=y_lab, showfliers=False,
                 hue=hue, order=tmp_order)
                 
ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('', fontsize=12)
ax.set_title('', fontsize=20)
ax.grid(False)

ax.tick_params(axis='x', labelrotation=45)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
x_lab = 'cluster'
y_lab = 'value'
hue = 'type'
col = 'donor'

g = sns.catplot(
    data=merged,
    x=x_lab, y=y_lab, hue=hue,
    col=col,                # facet by donor
    kind="box",
    order=tmp_order,
    showfliers=False,
    col_wrap=2,
    height=2, aspect=1.1,     # controls size/aspect of each facet
)

# Customize labels
# g.set_titles(col_template="{col_name}")  # each subplot titled by donor

# Rotate x-ticks
for i,ax in enumerate(g.axes.flat):
    s = donors[i]
    ax.tick_params(axis='x', labelrotation=45)
    ax.grid(False)
    ax.set_title(s, size=10)
    ax.set_ylim(0,1)

# Move legend outside
g._legend.set_bbox_to_anchor((1, 1))
plt.tight_layout(rect=[0, 0, 0.85, 1])  # leave space for legend

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
merged.to_csv(f'{outdir}/pmds_comp_rna_cluster_donor_mcg.csv', sep='\t')

In [ ]:
!date